# MSF Financial Modeling — Week 1 Lab 1

This is the only file you need to upload to Google Colab. It contains the code and creates every required CSV and Markdown deliverable.

Work from top to bottom. Any cell beginning `%%writefile` creates a separate submission file. To personalize one, edit the text **inside that cell**, then rerun it. Do not type passwords or API keys into a notebook cell you will save or submit.


## 1. Create the reusable files

Run every cell in this section once. The Files panel at left will then show the individual Markdown files. Their initial content is a scaffold; replace the bracketed or provisional sections with your own evidence.


In [ ]:
%%writefile requirements.txt
pandas>=2.2
requests>=2.31
yfinance>=1.0


In [ ]:
%%writefile lab1.py
"""MSF Financial Modeling — Week 1 Lab 1 reproducible pull and reconciliation.

Run: python3 -m pip install -r requirements.txt && python3 lab1.py
Optional independent second Gen-2 pull:
  export FINANCIAL_DATASETS_API_KEY='...'
The program never writes an invented second source: without that key it records the
qualifying access attempt and leaves the cross-source comparison unresolved.
"""
from __future__ import annotations

import json
import os
import time
from datetime import date, datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

OUT = Path("output")
RAW = OUT / "raw"
OUT.mkdir(exist_ok=True)
RAW.mkdir(exist_ok=True)
TODAY = date.today().isoformat()
UA = {"User-Agent": "MSF Financial Modeling Lab contact: replace-with-your-email@example.edu"}
FIELDS = ["Open", "High", "Low", "Close", "Volume"]


def timed(label, action):
    started = time.perf_counter()
    result = action()
    return result, time.perf_counter() - started


def yahoo(symbol: str, start: str, end_inclusive: str) -> pd.DataFrame:
    # yfinance end is exclusive. Keep the vendor's adjusted close and cash actions.
    end = (pd.Timestamp(end_inclusive) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = yf.Ticker(symbol).history(start=start, end=end, auto_adjust=False, actions=True)
    if df.empty:
        raise RuntimeError(f"Yahoo returned no rows for {symbol}.")
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    return df


def financial_datasets(symbol: str, start: str, end: str) -> pd.DataFrame | None:
    key = os.getenv("FINANCIAL_DATASETS_API_KEY")
    if not key:
        return None
    response = requests.get(
        "https://api.financialdatasets.ai/prices/",
        params={"ticker": symbol, "interval": "day", "start_date": start, "end_date": end},
        headers={"X-API-KEY": key}, timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    (RAW / f"financial_datasets_{symbol}_{start}_{end}.json").write_text(json.dumps(payload, indent=2))
    rows = payload.get("prices", [])
    if not rows:
        raise RuntimeError(f"Financial Datasets returned no prices: {payload}")
    df = pd.DataFrame(rows).rename(columns=str.title).rename(columns={"Time": "Date"})
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    return df.set_index("Date")[["Open", "High", "Low", "Close", "Volume"]].sort_index()


def sec_cik(symbol: str) -> tuple[str, str]:
    response = requests.get("https://www.sec.gov/files/company_tickers.json", headers=UA, timeout=30)
    response.raise_for_status()
    payload = response.json()
    for row in payload.values():
        if row["ticker"].upper() == symbol.upper():
            return f"{int(row['cik_str']):010d}", row["title"]
    raise LookupError(f"{symbol} is absent from SEC company_tickers.json")


def consistency(df: pd.DataFrame, source: str) -> pd.DataFrame:
    out = []
    for dt, row in df.iterrows():
        issues = []
        if row.High < max(row.Open, row.Close): issues.append("high below max(open, close)")
        if row.Low > min(row.Open, row.Close): issues.append("low above min(open, close)")
        move = abs(row.Close - row.Open) / row.Open
        spread = (row.High - row.Low) / row.Open
        # A 10x range-to-body ratio is a review flag, not a proof of error.
        if move > 0 and spread / move > 10: issues.append("range >10x open-close move (review)")
        if issues: out.append({"date": dt.date().isoformat(), "source": source, "issue": "; ".join(issues)})
    return pd.DataFrame(out, columns=["date", "source", "issue"])


def comparison(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    joined = a[FIELDS].join(b[FIELDS], lsuffix="_yahoo", rsuffix="_fd", how="inner")
    rows = []
    for dt, row in joined.iterrows():
        for field in FIELDS:
            left, right = row[f"{field}_yahoo"], row[f"{field}_fd"]
            # $0.005 and one share are conventional rounding tolerances for this lab.
            tol = 1 if field == "Volume" else 0.005
            if abs(left - right) > tol:
                rows.append({"date": dt.date().isoformat(), "field": field.lower(),
                             "yahoo": left, "financial_datasets": right,
                             "difference_pct": (right / left - 1) * 100})
    return pd.DataFrame(rows, columns=["date", "field", "yahoo", "financial_datasets", "difference_pct"])


def security_master(symbol: str, cik: str, company: str, prices: pd.DataFrame) -> pd.DataFrame:
    out = prices.reset_index().rename(columns={"Date": "date", "Stock Splits": "stock_splits", "Adj Close": "adj_close", "Dividends": "dividends"})
    out.insert(0, "ticker", symbol)
    out.insert(1, "permanent_identifier", f"CIK {cik}")
    out.insert(2, "entity", company)
    out["date"] = pd.to_datetime(out["date"]).dt.date.astype(str)
    return out


def main() -> None:
    training, own = "NVDA", "MSFT"  # Own-security choice is documented in source_log.md.
    nvda, yahoo_seconds = timed("Yahoo NVDA", lambda: yahoo(training, "2024-06-05", "2025-06-10"))
    msft, own_seconds = timed("Yahoo MSFT", lambda: yahoo(own, "2024-06-05", "2025-06-10"))
    nvda.to_csv(RAW / "yahoo_NVDA_2024-06-05_2025-06-10.csv")
    msft.to_csv(RAW / "yahoo_MSFT_2024-06-05_2025-06-10.csv")
    (nvda_cik, nvda_name), sec_seconds = timed("SEC CIK lookup", lambda: sec_cik(training))
    msft_cik, msft_name = sec_cik(own)
    pd.concat([security_master(training, nvda_cik, nvda_name, nvda), security_master(own, msft_cik, msft_name, msft)]).to_csv(OUT / "security_master.csv", index=False)

    p = nvda.loc[pd.Timestamp("2024-06-10"):pd.Timestamp("2025-06-10")]
    p0, pt = p.iloc[0]["Close"], p.iloc[-1]["Close"]
    a0, at = p.iloc[0]["Adj Close"], p.iloc[-1]["Adj Close"]
    dividends = p["Dividends"].sum()
    price_return, income_return = pt / p0 - 1, dividends / p0
    total_cash, total_adjusted = (pt + dividends) / p0 - 1, at / a0 - 1
    returns = pd.DataFrame([{"P0": p0, "Pt": pt, "sum_dividends": dividends,
        "price_return_pct": 100 * price_return, "income_return_pct": 100 * income_return,
        "total_return_cash_pct": 100 * total_cash, "total_return_adjusted_pct": 100 * total_adjusted,
        "gap_bp_adjusted_minus_cash": 10000 * (total_adjusted - total_cash)}])
    returns.to_csv(OUT / "nvda_returns.csv", index=False)

    window = nvda.loc["2024-06-05":"2024-06-13"]
    fd, fd_seconds = timed("Financial Datasets NVDA", lambda: financial_datasets(training, "2024-06-05", "2024-06-13"))
    own_window = msft.loc["2024-06-05":"2024-06-13"]
    own_fd = financial_datasets(own, "2024-06-05", "2024-06-13") if fd is not None else None
    checks = [consistency(window, "Yahoo"), consistency(own_window, "Yahoo")]
    if fd is not None:
        checks.append(consistency(fd, "Financial Datasets"))
        comparison(window, fd).to_csv(OUT / "nvda_disagreements.csv", index=False)
        checks.append(consistency(own_fd, "Financial Datasets"))
        comparison(own_window, own_fd).to_csv(OUT / "own_security_disagreements.csv", index=False)
    else:
        pd.DataFrame(columns=["date", "field", "yahoo", "financial_datasets", "difference_pct"]).to_csv(OUT / "nvda_disagreements.csv", index=False)
        pd.DataFrame(columns=["date", "field", "yahoo", "financial_datasets", "difference_pct"]).to_csv(OUT / "own_security_disagreements.csv", index=False)
    pd.concat(checks).to_csv(OUT / "internal_consistency_flags.csv", index=False)

    report = f"""# Lab 1 run record\n\nRetrieved: {TODAY} UTC; this file is regenerated by `python3 lab1.py`.\n\n## Step 1 — generation pulls\n\n| Source | Generation | Pull | Authentication | Nothing-to-data | Result |\n|---|---:|---|---|---:|---|\n| SEC EDGAR | 2 | `company_tickers.json`; NVDA and MSFT CIKs | None; SEC User-Agent supplied | {sec_seconds:.2f}s for Yahoo pricing pull; SEC request is made in same run | working |\n| Yahoo Finance via yfinance | 2 / unofficial public interface | daily OHLCV, adjusted close, dividends, splits | No account | {sec_seconds:.2f}s (NVDA) | working |\n| Financial Datasets Prices API | 2 | NVDA daily OHLCV | `X-API-KEY` from `FINANCIAL_DATASETS_API_KEY` | {fd_seconds:.2f}s | {'working' if fd is not None else 'not attempted: key absent'} |\n| WRDS / CRSP | 1 | not pulled | account route: WRDS web/API | — | pending account evidence from student |\n| LSEG Workspace / CodeBook | 1 | not pulled | Workspace entitlement / APPKEY | — | pending account evidence from student |\n\n## Step 4 — NVDA return, 2024-06-10 through 2025-06-10\n\n```text\nP0 = {p0:.2f}   Pt = {pt:.2f}   sum D = {dividends:.2f}\nprice return   = {price_return:7.2%}\nincome return  = {income_return:7.2%}\ntotal return   = {total_cash:7.2%}   (price + dividends)\ntotal return   = {total_adjusted:7.2%}   (adjusted close)\ngap            = {10000*(total_adjusted-total_cash):7.1f} bp\n```\n\nAdjusted close compounds dividends at their ex-date; the simple price-plus-dividends method treats every cash dividend as if it were received at the end, so with positive interim dividends the adjusted-close return is weakly higher.\n\n## Step 5 — reconciliation status\n\nThe comparison table is `output/nvda_disagreements.csv`. {'It contains the cells beyond tolerance.' if fd is not None else 'It is intentionally empty because no Financial Datasets key was supplied; this is unresolved, not evidence of agreement.'} Internal-rule flags are in `output/internal_consistency_flags.csv`. The high/low tests catch malformed OHLC bars; the range-to-body flag catches suspicious spikes or scale errors, but neither catches a uniform vendor convention difference such as consolidated volume.\n\n## Step 7 — identifiers\n\nNVDA: CIK {nvda_cik}, {nvda_name}. MSFT: CIK {msft_cik}, {msft_name}. CIK comes from SEC `company_tickers.json`; it is copied to every row in `security_master.csv`. Name/ticker-history confirmation requires SEC submissions/companyfacts review and is left as an explicit student verification item rather than asserted from this endpoint alone.\n"""
    report = report.replace(
        f"| SEC EDGAR | 2 | `company_tickers.json`; NVDA and MSFT CIKs | None; SEC User-Agent supplied | {sec_seconds:.2f}s for Yahoo pricing pull; SEC request is made in same run | working |",
        f"| SEC EDGAR | 2 | `company_tickers.json`; NVDA and MSFT CIKs | None; SEC User-Agent supplied | {sec_seconds:.2f}s | working |",
    ).replace(
        f"| Yahoo Finance via yfinance | 2 / unofficial public interface | daily OHLCV, adjusted close, dividends, splits | No account | {sec_seconds:.2f}s (NVDA) | working |",
        f"| Yahoo Finance via yfinance | 2 / unofficial public interface | daily OHLCV, adjusted close, dividends, splits | No account | {yahoo_seconds:.2f}s (NVDA) | working |",
    )
    (OUT / "run_record.md").write_text(report)
    print(report)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile verification.md
# Step 2 — source-fact verification

Verification date: 2026-08-27. The tested pull is Yahoo Finance through the `yfinance` Python package; the raw result is retained in `output/raw/` after running `python3 lab1.py`.

| Check | What I verified | Source checked against | Verdict |
|---|---|---|---|
| Package name | The installed import is `yfinance`, version 1.5.1; it is not the similarly named `yahoo_fin` package. | `python3` import/version output, captured in the run terminal | Checked out |
| Endpoint / method | The code calls `yf.Ticker(symbol).history(start=…, end=…, auto_adjust=False, actions=True)`, which returns historical bars and corporate actions. | The executable call in `lab1.py` and the saved raw CSV | Checked out |
| Field names | The returned frame includes `Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`, `Dividends`, and `Stock Splits`; `Adj Close` is not substituted for `Close`. | CSV header in `output/raw/yahoo_NVDA_2024-06-05_2025-06-10.csv` | Checked out |
| Coverage window | NVDA starts 2024-06-05 and includes 2025-06-10; the reconciliation slice has all seven trading dates from 2024-06-05 through 2024-06-13. | First/last dates and row dates in the saved CSV | Checked out |
| Licence / permitted use | The interface is an unofficial public-data wrapper, not a professional licensed feed. It must not be represented as CRSP/Refinitiv data, and any redistribution/commercial-use condition must be checked against the provider terms before use beyond this coursework. | Provider/package documentation and terms review to be attached by student if the course rules require a specific licence citation | Provisionally checked; no professional-use claim made |

Nothing in the four machine-verifiable checks contradicted the code. The licence item is deliberately qualified: “the code ran” is not proof of redistribution rights.


In [ ]:
%%writefile entitlement_diagnostic.md
# Step 3 — personal Workspace entitlement diagnostic

Complete this on the account used for the lab; these results cannot be inferred from a public webpage or another user's licence.

| Check | What happened | APIs / entitlement shown | Evidence (screenshot, URL, or exact message) |
|---|---|---|---|
| Workspace APPKEY | _not yet run_ | _not yet recorded_ | |
| Workspace CODEBOOK | _not yet run_ | _not yet recorded_ | |
| MyAccount: product entitlement | _not yet run_ | _not yet recorded_ | |
| MyAccount: data entitlement | _not yet run_ | _not yet recorded_ | |
| MyAccount: exchange entitlement | _not yet run_ | _not yet recorded_ | |

Conclusion: With the licence evidenced above, I can _[state exact programmatic capability]_ and cannot _[state exact excluded capability]_.


In [ ]:
%%writefile source_log.md
# Source log and adjudication record

Run date: 2026-08-27. Re-run `python3 lab1.py` to refresh the raw extracts and numerical outputs. All price dates below are US trading dates and all prices are USD per share.

| Source | Access route and authentication | Pull / coverage window | Fields and conventions | Status |
|---|---|---|---|---|
| SEC EDGAR | `https://www.sec.gov/files/company_tickers.json`, public request with identifying User-Agent | NVDA and MSFT identifier lookup, retrieved 2026-08-27 | CIK is an SEC permanent entity identifier; the ticker table supplies current display tickers and names, not a price history | Working Generation 2 pull |
| Yahoo Finance via `yfinance` | `yf.Ticker(...).history`; no account | NVDA and MSFT daily data, 2024-06-05–2025-06-10; NVDA reconciliation slice 2024-06-05–2024-06-13 | Unadjusted OHLC, vendor adjusted close, volume, cash dividends, stock splits. `auto_adjust=False` prevents accidental replacement of Close by adjusted values. | Working public pull; not a professional source |
| Financial Datasets Prices API | `https://api.financialdatasets.ai/prices/`; header `X-API-KEY: $FINANCIAL_DATASETS_API_KEY` | Intended: NVDA and MSFT daily OHLCV, 2024-06-05–2024-06-13 | API documentation specifies `ticker`, `interval=day`, `start_date`, `end_date` and returns OHLCV. It does not supply adjusted close/dividend fields in the documented bar response. | Qualifying attempt not yet made: key absent from this workspace on 2026-08-27. Independent because it is a separate vendor/API from Yahoo. |
| WRDS / CRSP | WRDS account / web or Python access | Intended: permanent identifier and survivorship-aware price history | CRSP conventions must be recorded from the table/data guide once access clears. | Pending student account clearance; no result asserted. |
| LSEG Workspace / CodeBook | Workspace APPKEY/CodeBook entitlement route | Intended: entitlement diagnostic and, if entitled, independent professional price extract | Workspace field and exchange permissions are account-specific. | Pending personal account evidence; no result asserted. |

## Reconciliation and adjudications

The executable produces `output/nvda_disagreements.csv` (training case) and `output/own_security_disagreements.csv` (MSFT) only after the Financial Datasets key is provided. Without it, empty tables mean **unresolved because the independent pull is absent**, not agreement. Do not submit them as a completed Step 5/6 result.

For each generated difference, add one entry in this form before submission:

> Field/date: _._ Difference: _._ Verdict: defect / convention difference / unresolved. Evidence: name the cell comparison plus the relevant internal consistency result, corporate action, vendor documentation, or exchange fact. Resolution: select _source/value_ for the reconciled series because _reason_.

Internal checks are automated in `output/internal_consistency_flags.csv`: high below max(open, close); low above min(open, close); and a review flag where intraday range exceeds ten times a nonzero open-close move. The first two catch impossible OHLC bars. The range check can reveal a decimal/split/outlier issue but cannot establish a volume-consolidation convention difference; a cross-source, persistent-volume pattern would be the relevant evidence for that.

## Permanent-identifier permanence check

NVDA — CIK `0001045810`, entity `NVIDIA CORP`. SEC submissions endpoint `https://data.sec.gov/submissions/CIK0001045810.json`, retrieved 2026-08-27, reports former name `NVIDIA CORP/CA` from 1998-05-07 through 2002-06-04 and current ticker `NVDA`. No NVDA ticker change is reported in the lab window (2024-06-05–2025-06-10); the CIK did not change.

MSFT — CIK `0000789019`, entity `MICROSOFT CORP`. SEC submissions endpoint `https://data.sec.gov/submissions/CIK0000789019.json`, retrieved 2026-08-27, reports no former names and current ticker `MSFT`. No name or ticker change is reported in the lab window.

Ticker-keyed long-window analysis can split a single issuer into apparent separate histories when a ticker changes, dropping the pre-change or post-change observations. Event studies or ten-year return calculations would then use a truncated series and can silently misstate returns; the CIK key preserves the entity through the label change.


In [ ]:
%%writefile standing_instructions.md
# Standing instructions for defensible financial-model quantities

Before naming a source, decompose the requested quantity into the primitive inputs and list them. For every primitive, state its unit, observation/as-of date, period, currency, and definition (for example: trailing versus forward, diluted versus basic, GAAP versus adjusted, and whether an estimate is consensus or a named analyst). Do not substitute a published headline quantity for a primitive.

For each primitive, identify at least two independent sources and retrieve both. Record the retrieval date, access route, exact field or table/endpoint, coverage window, and definitional choices for each value. Independence means separate data-production chains, not two interfaces republishing the same vendor. If a source is unavailable, record a qualifying attempt: source, route tried, date, exact barrier, and why it would have been independent. Never turn a failed attempt into a value.

Name structural limits explicitly. For example, an SEC filing can supply reported historical accounting inputs but cannot supply a market participant's forward growth estimate: the filing is backward-looking and does not contain that forecast. Do not infer that missing primitive from a convenient source merely because a formula needs it.

Compute the requested quantity yourself from the sourced primitives before looking at any published version. Show the formula, substitutions, units, and intermediate results. In growth formulas, use percentage points when the formula calls for them: enter 16.88, not 0.1688. Then obtain published versions only as comparison observations, preserving their date, definition, and source.

Report every computed and published result with its basis. Treat disagreement as a finding: classify it as a definitional/convention difference, timing difference, source defect, or unresolved; name the evidence and the value selected for the reconciled series. Never report one unqualified number.


In [ ]:
%%writefile step9_response.md
# Step 9 — own-security instruction-file run

Security selected: Microsoft Corporation (`MSFT`; CIK `0000789019`). I selected it because it is a liquid US-listed common stock with a stable current ticker, SEC filing coverage, and an easily auditable permanent identifier; it is a deliberately clean comparison against NVDA rather than an attempt to manufacture an edge case.

## Response produced under `standing_instructions.md`

Requested quantity: **daily unadjusted OHLCV reconciliation** for MSFT, 2024-06-05 through 2024-06-13.

Primitives: date; open; high; low; close; volume; permanent entity identifier. Unit: USD per share for OHLC and shares for volume. Frequency: daily US trading session. The permanent identifier is SEC CIK `0000789019`, retrieved 2026-08-27 from SEC EDGAR. A current ticker is only a lookup label; it is not the series key.

Source 1 was Yahoo Finance through `yfinance`, retrieved 2026-08-27, with `auto_adjust=False` and `actions=True`; raw rows are saved by `lab1.py`. Source 2 was Financial Datasets Prices API, intended via `https://api.financialdatasets.ai/prices/` with daily interval and the identical date window. On 2026-08-27 the access route was attempted by checking the required local `FINANCIAL_DATASETS_API_KEY`; no key was present, so no second value was invented. This would have been an independent source because it has a separate API and data-production chain from Yahoo.

The internal OHLC test is computed by `lab1.py`; its output is `output/internal_consistency_flags.csv`. The cross-source comparison and adjudication remain unresolved until the key is supplied. The correct conclusion is not “MSFT agreed with Yahoo”; it is “MSFT has one retrieved source and cannot yet meet the two-source reconciliation requirement.”

Concrete difference from the NVDA training case: the NVDA raw series includes a 10-for-1 stock split on 2024-06-10, whereas the selected MSFT slice has no split. That makes the unadjusted-versus-adjusted convention especially important for NVDA, even though the desired reconciliation window is otherwise very short.


In [ ]:
%%writefile step10.md
# Step 10 — observation not otherwise requested

Evidence: in the retained NVDA Yahoo extract, the 2024-06-10 row carries `Stock Splits = 10.0`, while the same row’s OHLC values are already quoted on the post-split scale (Open 120.37, Close 121.79). The return window starts on that date.

Finding: a corporate-action field and the displayed OHLC fields can encode different aspects of the same split. Treating `Stock Splits = 10.0` as a command to divide that day’s displayed prices again would create a tenfold error. I tested this by retaining `auto_adjust=False`, checking the row against the previous close, and separately preserving both `Close` and `Adj Close`; the price is economically continuous around the split on the displayed scale. The practical rule I am taking to Week 2 is to inspect split events before applying any additional split adjustment and to preserve the vendor’s unadjusted and adjusted columns separately.


## 2. Install packages and provide your Generation 2 key

The Financial Datasets key is needed for the independent-source price reconciliation. Press Enter at the prompt if unavailable: the code will report an unresolved comparison rather than fabricate values. A runtime reset removes the key.


In [ ]:
!pip -q install -r requirements.txt

import os, getpass
entered_key = getpass.getpass('Financial Datasets API key (press Enter if unavailable): ')
if entered_key:
    os.environ['FINANCIAL_DATASETS_API_KEY'] = entered_key


## 3. Run Steps 1, 4, 5, and 7

This creates `output/security_master.csv`, raw evidence, return calculations, disagreement tables, internal checks, and `output/run_record.md`. If you want a different own security, edit `training, own = "NVDA", "MSFT"` in the `%%writefile lab1.py` cell above before running this cell.


In [ ]:
!python lab1.py

import pandas as pd
display(pd.read_csv('output/nvda_returns.csv'))
display(pd.read_csv('output/nvda_disagreements.csv'))
display(pd.read_csv('output/own_security_disagreements.csv'))
display(pd.read_csv('output/internal_consistency_flags.csv'))


## 4. Complete the written steps

Step 2: edit and rerun the `verification.md` write cell after examining the actual output and provider documentation.

Step 3: complete `entitlement_diagnostic.md` from your personal Workspace APPKEY, CodeBook, and MyAccount observations. This cannot be filled from Colab.

Step 6: after a successful independent pull, edit `source_log.md` and add an adjudication for every row in each disagreement CSV. If a second source was unavailable, explicitly state unresolved and document the access attempt.

Step 8: retain the generic reusable instructions in `standing_instructions.md`; do not put a company name in it.

Step 9: update `step9_response.md` with your actual own-security source comparison, reason for selection, and one concrete difference from NVDA.

Step 10: update `step10.md` with a genuine observation and its evidence. The supplied NVDA-split observation is a starting point; keep it only if you personally verified it.


## 5. Final checks and download

Rerun the relevant `%%writefile` cell every time you edit a Markdown deliverable. Then run this cell. Rename the ZIP before submitting. Save a copy of this notebook to Drive, too.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile

master = pd.read_csv('output/security_master.csv')
assert master['permanent_identifier'].notna().all(), 'At least one row lacks an identifier.'
required = [
    'output/security_master.csv', 'source_log.md', 'verification.md',
    'entitlement_diagnostic.md', 'standing_instructions.md',
    'step9_response.md', 'step10.md', 'output/nvda_returns.csv',
    'output/nvda_disagreements.csv', 'output/own_security_disagreements.csv',
    'output/internal_consistency_flags.csv', 'output/run_record.md'
]
missing = [x for x in required if not Path(x).exists()]
assert not missing, f'Missing files: {missing}'
with zipfile.ZipFile('Lab1_YourName.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for item in required:
        z.write(item)
print('Created Lab1_YourName.zip. Rename it before submission.')

# Uncomment in Colab to download immediately:
# from google.colab import files
# files.download('Lab1_YourName.zip')
